# Comprehensive COVID-19 Pandemic Data Analysis

This notebook uses a reusable pipeline for schema normalization, date extraction, data cleaning, duplicate checks, country-level aggregation, and time-series analysis.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    discover_csv_files, inspect_schema, load_covid_data,
    missing_data_report, clean_covid_data, duplicate_report,
    remove_exact_duplicates, aggregate_country_daily,
    add_daily_changes, get_latest_country_data,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'raw'


## 1. Schema and Report-Date Inspection
The report date is extracted primarily from each CSV filename. `Last_Update` remains available as a timestamp column and is not used as a replacement for the report date unless a filename date cannot be extracted.


In [ ]:
csv_files = discover_csv_files(DATA_DIR)
schema_report = inspect_schema(csv_files)
date_report = pd.DataFrame(
    {'Source_File': list(schema_report['dates_from_filenames'].keys()),
     'Extracted_Date': list(schema_report['dates_from_filenames'].values())}
)
display(date_report)
print('Files with no extracted filename date:', date_report['Extracted_Date'].isna().sum())


## 2. Load and Standardize Data


In [ ]:
df = load_covid_data(DATA_DIR)
print('Raw standardized shape:', df.shape)
print('Date range:', df['Date'].min(), 'to', df['Date'].max())
display(df.head())


## 3. Missing Data Strategy
Missing values are reported before and after conservative cleaning. Unknown values are not blindly replaced with zero. `Active` is derived only when Confirmed, Deaths, and Recovered are available.


In [ ]:
missing_before = missing_data_report(df)
display(missing_before)

df = clean_covid_data(df, compute_active=True)
missing_after = missing_data_report(df)
display(missing_after)


## 4. Duplicate Analysis
Exact duplicate rows are removed. Key-based duplicate counts are reported for investigation but are not automatically removed because repeated administrative keys may require domain-specific validation.


In [ ]:
duplicates_before = duplicate_report(df)
print(duplicates_before)

df = remove_exact_duplicates(df)
duplicates_after = duplicate_report(df)
print(duplicates_after)


## 5. Country-Level Daily Time Series
Subnational records are aggregated into country totals for each report date before calculating temporal trends.


In [ ]:
country_daily = aggregate_country_daily(df)
country_daily = add_daily_changes(country_daily)
display(country_daily.head())


## 6. Global Time Series


In [ ]:
global_daily = (
    country_daily.groupby('Date', as_index=False)
    [[c for c in ['Confirmed', 'Deaths', 'Recovered', 'Active'] if c in country_daily.columns]]
    .sum(min_count=1)
)
display(global_daily.tail())

plot_columns = [c for c in ['Confirmed', 'Deaths', 'Recovered', 'Active'] if c in global_daily.columns]
global_daily.set_index('Date')[plot_columns].plot(figsize=(12, 6))
plt.title('Global COVID-19 Cumulative Trends')
plt.xlabel('Date')
plt.ylabel('Cases')
plt.tight_layout()
plt.show()


## 7. Latest Available Data by Country
The latest date is determined separately for every country, then country-level regional totals are returned. This avoids incorrectly selecting the same global date for countries whose reporting ended earlier.


In [ ]:
latest_country_data = get_latest_country_data(df)
display(latest_country_data.head(20))


## 8. Data Quality Summary


In [ ]:
print('Final cleaned record count:', len(df))
print('Country-date records:', len(country_daily))
print('Countries:', country_daily['Country_Region'].nunique())
print('Date range:', country_daily['Date'].min(), 'to', country_daily['Date'].max())
